# FeFET 增量编程 ISPP (52) · 一次擦除 → 递增 Vg → 每步读 · D 恒 0.1V

> 2026-06-11 建。按椰椰时序图：**只 reset 一次**（开头一发 −V 擦除）→ Vg 逐步增加,
> 每加一次在 Vg=0.5V 读一次（D 始终 0.1V、S 0V）,程序间**不再擦除**（累积/增量编程）。
> 出 **Id-vs-累积编程幅值** 曲线。
>
> ⚠️ 与 50（MLC,每个幅值都擦除复位）**不同**：52 全程只擦一次、增量加压。
> 这是**自定义波形 notebook**——波形代码就在下面 `run_ispp` 里,**看得见、可改**;
> 复用了协议里验证过的读窗解析/会话初始化,降低手搓风险。波形稳定后可并入引擎。

## 安全
- 先 `LIVE=False` dry 跑通（无硬件,审计波形）,再 `LIVE=True` 上真机。
- 读量程默认 `1UA`（覆盖 1~200nA;协议默认 100UA 对 nA 太粗）。
- 盯 |Ig| 图:读电压下异常抬升=栅漏,停手查接线/降压。脆弱器件先用短 `VG_LIST`、小幅值。

In [ ]:
# ===================== 【配置】改这里 =====================
DEVICE_ID, GEOMETRY, SERIAL = "qzb_sy2", "L300W300", "1"
DEVICE_TYPE, OPERATOR = "FeFET", "yhzang"
GATE_CH, DRAIN_CH = 202, 201          # WGFMU 通道:Gate=202 / Drain=201

LIVE = False                          # 先 dry 跑通,再 True 上真机

# ISPP 时序(按椰椰时序图)
V_ERASE     = 4.0                     # 开头一次擦除幅值(绝对值,实际打 -4V)
ERASE_WIDTH = 50e-6                   # 擦除脉宽 s
VG_LIST     = [1.5, 2.0, 2.5, 3.0, 3.5, 3.8]  # 递增编程幅值 V(从小到大)
PROG_WIDTH  = 50e-6                   # 每发编程脉宽 s(50µs)
VG_READ     = 0.5                     # 读取 Vg V(脉冲间的基线)
VD          = 0.1                     # Drain 恒定 V(全程)
SETTLE_S    = 10e-6                   # 脉冲后→读 之间的落座沉降 s（让栅极位移瞬态衰减,关键!大器件尤其要)
T_READ      = 5e-6                    # 单次读窗宽 s
N_PTS       = 5                       # 读窗硬件平均点数
IRANGE_DRAIN = "10UA"                 # Drain 读量程;实测 on 态 ~µA,1UA 会过量程;读不出小电流再降
IRANGE_GATE  = "1MA"                  # Gate 量程(栅漏监看)
# =========================================================
print(f"ISPP {DEVICE_ID}/{GEOMETRY}#{SERIAL}: 擦除 -{V_ERASE}V 一次 → 递增 {VG_LIST}V"
      f"(各 {PROG_WIDTH*1e6:g}µs) → 读 Vg{VG_READ}/Vd{VD}  量程{IRANGE_DRAIN}  LIVE={LIVE}")

In [ ]:
import sys, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "fefetlab").exists())
sys.path.insert(0, str(ROOT / "src"))

from fefetlab.protocols.wgfmu_fefet import (
    parse_args, make_backend, configure_channel_map,
    _summarize_windows, _ensure_wgfmu_initialized, _safe_disconnect, T_RF,
)
from fefetlab.orchestration.core import ExperimentContext


def run_ispp(backend):
    # === 椰椰时序:settle@读电平 → 一次擦除 → 读(erased) → [递增编程 Vk → 读] × N ===
    # gp=栅极波形(脉冲),dp=漏极(全程恒 VD)。读窗 t0/t1 约定与协议 _build_read_phase 一致。
    g = lambda dt, v: backend.add_vector("gp", dt, float(v))   # 栅极加一段
    d = lambda dt, v: backend.add_vector("dp", dt, float(v))   # 漏极加一段(恒 VD)
    backend.clear()
    backend.create_pattern("gp", 0.0)
    backend.create_pattern("dp", 0.0)

    guard = min(200e-9, T_READ * 0.2)
    meas_window = max(T_READ - guard, T_READ * 0.5)
    interval = meas_window / max(N_PTS, 1)
    average = min(200e-9, interval * 0.8)

    t = 0.0
    windows = []

    def add_read(prog_v):
        # 脉冲后先落座沉降 SETTLE_S(让栅极位移/弛豫瞬态衰减),再起 T_READ 读窗,登记测量窗
        nonlocal t
        if SETTLE_S > 0:
            g(SETTLE_S, VG_READ); d(SETTLE_S, VD); t += SETTLE_S
        rs = t
        windows.append({"idx": len(windows), "vg": VG_READ, "vd": VD,
                        "t0": rs + guard, "t1": rs + T_READ, "prog_v": prog_v})
        g(T_READ, VG_READ); d(T_READ, VD); t += T_READ

    g(1e-3, VG_READ); d(1e-3, VD); t += 1e-3                      # settle 在读电平
    g(T_RF, -abs(V_ERASE)); d(T_RF, VD); t += T_RF               # ↓ 擦除(只此一次)
    g(ERASE_WIDTH, -abs(V_ERASE)); d(ERASE_WIDTH, VD); t += ERASE_WIDTH
    g(T_RF, VG_READ); d(T_RF, VD); t += T_RF                     # 回读电平
    add_read(0.0)                                               # 读 erased 态
    for vk in VG_LIST:                                          # 递增编程,程序间不擦除
        g(T_RF, vk); d(T_RF, VD); t += T_RF
        g(PROG_WIDTH, vk); d(PROG_WIDTH, VD); t += PROG_WIDTH    # ↑ 编程脉冲(脉冲只加栅极)
        g(T_RF, VG_READ); d(T_RF, VD); t += T_RF                 # 回读电平
        add_read(vk)

    for w in windows:
        backend.set_measure_event("gp", f"g{w['idx']}", w["t0"], N_PTS, interval, average, "averaged")
        backend.set_measure_event("dp", f"d{w['idx']}", w["t0"], N_PTS, interval, average, "averaged")

    # 配置 + 跑(自定义量程;结构同协议 _configure_and_run_phase,AuditBackend 也支持)
    backend.add_sequence(GATE_CH, "gp", 1)
    backend.add_sequence(DRAIN_CH, "dp", 1)
    _ensure_wgfmu_initialized(backend)
    for ch, rng in ((GATE_CH, IRANGE_GATE), (DRAIN_CH, IRANGE_DRAIN)):
        backend.set_operation_mode(ch, "FASTIV")
        backend.set_force_voltage_range(ch, "AUTO" if ch == GATE_CH else "3V")
        backend.set_measure_enabled(ch, True)
        backend.set_measure_mode(ch, "CURRENT")
        backend.set_measure_current_range(ch, rng)
    backend.set_timeout(60.0)
    backend.connect(GATE_CH); backend.connect(DRAIN_CH)
    try:
        backend.execute(); backend.wait_until_completed()
        g_df = backend.get_measure_values(GATE_CH)
        d_df = backend.get_measure_values(DRAIN_CH)
    finally:
        _safe_disconnect(backend, GATE_CH, DRAIN_CH)

    rows = _summarize_windows(g_df, d_df, windows)              # 复用协议读窗解析
    for r, w in zip(rows, windows):
        r["prog_v"] = w["prog_v"]
        r["step"] = w["idx"]
    return pd.DataFrame(rows)


print("ISPP 工具就绪。波形见上面 run_ispp(可看可改)。")

## 跑（dry: LIVE=False 无硬件审计；live: 真机驱动 WGFMU）
器件已落针、接线确认后再 `LIVE=True`。

In [ ]:
_a = parse_args(["--gate-ch", str(GATE_CH), "--drain-ch", str(DRAIN_CH)])
configure_channel_map(_a)                 # 设通道全局(make_backend 要读)
backend, addr = make_backend(LIVE)
try:
    df = run_ispp(backend)
finally:
    try:
        backend.close_session()
    except Exception:
        pass

ctx = ExperimentContext(root=ROOT, device_id=DEVICE_ID, geometry=GEOMETRY, serial=SERIAL, live=LIVE)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = ROOT / "runs" / ctx.device_slug / ctx.die_slug / ("live" if LIVE else "dry") / f"{ts}_ISPP_incremental"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "ispp_incremental.csv", index=False, encoding="utf-8")
print(f"步数={len(df)}  |Id|max={df['Id_mean_A'].abs().max():.3e}A  |Ig|max={df['Ig_mean_A'].abs().max():.3e}A")
print(f"落盘: {out_dir}")
df[["step", "prog_v", "Id_mean_A", "Ig_mean_A", "n_d"]]

## 看图（Id-vs-累积编程幅值 + 栅漏监看）

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(df["prog_v"], df["Id_mean_A"].abs(), "o-")
ax.set_yscale("log")
ax.set_xlabel("编程幅值 Vg (V)  [0 = 擦除态]")
ax.set_ylabel("|Id| (A)  @ Vg=0.5/Vd=0.1")
ax.set_title(f"ISPP 增量编程  {DEVICE_ID} {GEOMETRY}#{SERIAL}")
ax.grid(True, which="both", alpha=.3)
plt.show()

fig2, ax2 = plt.subplots(figsize=(6.5, 2.6))
ax2.plot(df["prog_v"], df["Ig_mean_A"].abs(), "r.-")
ax2.set_yscale("log")
ax2.set_xlabel("编程幅值 (V)")
ax2.set_ylabel("|Ig| (A)")
ax2.set_title("栅漏监看 |Ig|")
ax2.grid(alpha=.3)
plt.show()